# 01 — Exploring Claude Code GitHub Issues

## Goal
Pull a sample of open issues and inspect the data shape before designing the categorization schema.

## 1. Setup

In [ ]:
%pip install -r ../requirements.txt

In [ ]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

from src.github_client import fetch_issues

REPO = 'anthropics/claude-code'
DATA_DIR = Path('../data')
DATA_DIR.mkdir(exist_ok=True)

## 2. Fetch a Sample

In [ ]:
issues = fetch_issues(REPO, state='open', limit=1000)
print(f'Fetched {len(issues)} issues')

## 3. Inspect the Data

In [ ]:
# Top-level fields and types
sample = issues[0]
print("Fields:", list(sample.keys()))
print()
for k, v in sample.items():
    print(f"  {k}: {type(v).__name__} — {repr(v)[:80]}")

In [ ]:
# One full issue — readable view
print(f"#{sample['number']} — {sample['title']}")
print(f"Author : {sample['user']['login']}")
print(f"Labels : {[l['name'] for l in sample['labels']]}")
print(f"Created: {sample['created_at']}")
print(f"Comments: {sample['comments']}")
print()
print("--- Body (first 500 chars) ---")
print((sample["body"] or "")[:500])

In [ ]:
# Label usage across the sample
from collections import Counter

all_labels = [l["name"] for issue in issues for l in issue["labels"]]
unlabeled = sum(1 for issue in issues if not issue["labels"])

print(f"Unlabeled issues: {unlabeled} / {len(issues)}")
print()
print("Label counts:")
for label, count in Counter(all_labels).most_common():
    print(f"  {count:>3}  {label}")

## 4. Save Raw Sample

In [ ]:
out = DATA_DIR / "raw_sample.json"
out.write_text(json.dumps(issues, indent=2))
print(f"Saved {len(issues)} issues → {out}")

## What I Learned

- **Fields available:** 30+ fields from the REST API. Most useful: `number`, `title`, `body`, `labels`, `user`, `comments`, `reactions`, `created_at`, `updated_at`, `html_url`. Note: `type` is always `null`; `reactions.total_count` duplicates the per-reaction sum.
- **Body quality:** Issues follow structured GitHub templates (headers like "Documentation Type", "Documentation Location") — makes body parsing tractable, though older or informal issues may not conform.
- **Label coverage:** All 5 sampled issues were labeled. `enhancement` appeared on all 5; `documentation` on 4 of 5. `area:*` labels are sparse but precise signal when present.
- **Surprises:** Pull requests appear in the `/issues` endpoint and must be filtered out (check for absence of `pull_request` key). The old `gh()` wrapper parsed `--json` fields but never applied filtering.
- **Decisions for categorization schema:** Labels are the primary signal — `area:*` for functional area, `platform:*` for OS/environment, `bug`/`enhancement`/`regression` for type. Title keyword fallback handles unlabeled issues. Priority combines label signals with `reactions + comments` engagement score.